In [199]:
import pandas as pd
import numpy as np
import ast

In [200]:
file = '../../../data/oriental_store/oriental_store_products.csv'

In [201]:
df = pd.read_csv(file, index_col=0)

In [302]:
def clean_specs(specs_string:dict):
    
    if specs_string == '{}' or pd.isna(specs_string):
        return ''


    try:
        specs_dict = ast.literal_eval(specs_string)

        key_specs = {'processor',
                    'memory',
                    'storage',
                    'graphic'}


        specs_list = []
        for k, v in specs_dict.items():
            k_set = set(k.lower().split())
            
            if not k_set.isdisjoint(key_specs) and 'number' not in k_set:
                specs_list.append(str(v).lower())

        cleaned_specs =  ', '.join(specs_list)

        return cleaned_specs 
        
    except:
        return ''
    

In [303]:
def clean_categories(categories_string:set):

    if categories_string == '{}' or pd.isna(categories_string):
        return ''

    try:
        categories_set = ast.literal_eval(categories_string)

        cleaned_categories = str(categories_set).replace('\'', '').lower()[1:-1]

        return cleaned_categories

    except Exception as e:
        print(e)
        return ''

In [393]:
def preprocess_file(file_path):

    df = pd.read_csv(file_path, index_col=0)

    df['in_stock'] = df['in_stock'].astype(bool) 
    valid_df = df[df['in_stock'] == True].copy()

    valid_df['price'] = valid_df['price'].astype(np.float64)
    valid_df['original_price'] = valid_df['original_price'].fillna(valid_df['price'])
    
    valid_df = valid_df.drop(columns=['in_stock','currency'])

    valid_df['product_name'] = valid_df['product_name'].apply(str.lower)

    valid_df['brand'] = valid_df['brand'].apply(str.lower)
    
    valid_df['specs'] = valid_df['specs'].apply(clean_specs)

    valid_df['categories'] = valid_df['categories'].apply(clean_categories)

    valid_df.loc[valid_df['product_description'] == 'Unknown','product_description'] = 'No Description'

    model_text = (valid_df['brand'] + ' ' +
                    valid_df['product_name'] + ' ' +
                    valid_df['specs'])
                   

    valid_df['model_text'] = model_text


    return valid_df

In [404]:
def combine_preprocess(paths_list):

    dfs = []
    for i in range(len(paths_list)):
        df = preprocess_file(paths_list[i])
        print(paths_list[i])
        print(df.loc[0,'store'])
        dfs.append(df)
        
    combined_df = pd.concat(dfs, axis=0, ignore_index=True)

    return combined_df

In [405]:
paths = ['../../data/compujordan/compujordan_products.csv', file]

In [406]:
ds = combine_preprocess(paths)

../../data/compujordan/compujordan_products.csv
Compu Jordan
../../data/oriental_store/oriental_store_products.csv
Oriental Store


In [407]:
ds['store'].unique()

array(['Compu Jordan', 'Oriental Store'], dtype=object)

In [408]:
ds.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2109 entries, 0 to 2108
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   product_code         2109 non-null   object 
 1   product_name         2109 non-null   object 
 2   brand                2109 non-null   object 
 3   store                2109 non-null   object 
 4   url                  2109 non-null   object 
 5   image_url            2109 non-null   object 
 6   price                2109 non-null   float64
 7   original_price       2109 non-null   float64
 8   product_description  1650 non-null   object 
 9   categories           2109 non-null   object 
 10  specs                2109 non-null   object 
 11  model_text           2109 non-null   object 
dtypes: float64(2), object(10)
memory usage: 197.8+ KB


In [399]:
j = pd.read_csv(paths[0])

In [400]:
j.head()

,Unnamed: 0,product_code,product_name,brand,store,url,image_url,price,original_price,currency,in_stock,product_description,categories,specs
0,0,Ultra 7 265K,Intel Core Ultra 7 265K – 20-Core Processor (B...,Intel,Compu Jordan,https://compujordan.com/intel-core-ultra-7-265...,https://compujordan.com/image/cache/catalog/pr...,309.0,NaN,JOD,True,The Intel Core Ultra 7 265K is a next-generati...,"{'Computer Hardware', 'Gaming Components', 'Ga...",{}
1,1,Z890 GAMING X WIFI7,GIGABYTE Z890 Gaming X Wi-Fi 7 & Bluetooth DDR...,GIGABYTE,Compu Jordan,https://compujordan.com/gigabyte-z890-gaming-x...,https://compujordan.com/image/cache/catalog/pr...,219.0,NaN,JOD,True,The GIGABYTE Z890 Gaming X Wi-Fi 7 ATX Motherb...,"{'Computer Hardware', 'Gaming Components', 'Ga...",{}
2,2,i7-14700KF,Intel Core i7-14700KF – 14th Gen Unlocked CPU ...,Intel,Compu Jordan,https://compujordan.com/intel-core-i7-14700kf-...,https://compujordan.com/image/cache/catalog/pr...,299.0,NaN,JOD,True,The Intel Core i7-14700KF is a 14th Gen Raptor...,"{'Computer Hardware', 'Gaming Components', 'Ga...",{}
3,3,Core Ultra 9 285K,"Intel Core Ultra 9 285K (Series 2) – 24-Core, ...",Intel,Compu Jordan,https://compujordan.com/intel-core-ultra-9-285...,https://compujordan.com/image/cache/catalog/pr...,569.0,NaN,JOD,True,The Intel Core Ultra 9 285K (Series 2) is one ...,"{'Computer Hardware', 'Gaming Components', 'Ga...",{}
4,4,RTX 5070 Ti 16G SHADOW 3X OC,MSI GeForce RTX 5070 Ti 16G SHADOW 3X OC – 16G...,MSI,Compu Jordan,https://compujordan.com/msi-geforce-rtx-5070-t...,https://compujordan.com/image/cache/catalog/pr...,789.0,NaN,JOD,True,The MSI GeForce RTX 5070 Ti 16G SHADOW 3X OC i...,"{'Computer Hardware', 'Gaming Components', 'Ga...",{}


In [401]:
j[j['price'] == 'False']

,Unnamed: 0,product_code,product_name,brand,store,url,image_url,price,original_price,currency,in_stock,product_description,categories,specs


In [402]:
c = preprocess_file(paths[0])

In [403]:
c.head()

,product_code,product_name,brand,store,url,image_url,price,original_price,product_description,categories,specs,model_text
0,Ultra 7 265K,intel core ultra 7 265k – 20-core processor (b...,intel,Compu Jordan,https://compujordan.com/intel-core-ultra-7-265...,https://compujordan.com/image/cache/catalog/pr...,309.0,309.0,The Intel Core Ultra 7 265K is a next-generati...,"gaming components, components, gaming, compute...",,intel intel core ultra 7 265k – 20-core proces...
1,Z890 GAMING X WIFI7,gigabyte z890 gaming x wi-fi 7 & bluetooth ddr...,gigabyte,Compu Jordan,https://compujordan.com/gigabyte-z890-gaming-x...,https://compujordan.com/image/cache/catalog/pr...,219.0,219.0,The GIGABYTE Z890 Gaming X Wi-Fi 7 ATX Motherb...,"gaming components, components, gaming, compute...",,gigabyte gigabyte z890 gaming x wi-fi 7 & blue...
2,i7-14700KF,intel core i7-14700kf – 14th gen unlocked cpu ...,intel,Compu Jordan,https://compujordan.com/intel-core-i7-14700kf-...,https://compujordan.com/image/cache/catalog/pr...,299.0,299.0,The Intel Core i7-14700KF is a 14th Gen Raptor...,"gaming components, components, gaming, compute...",,intel intel core i7-14700kf – 14th gen unlocke...
3,Core Ultra 9 285K,"intel core ultra 9 285k (series 2) – 24-core, ...",intel,Compu Jordan,https://compujordan.com/intel-core-ultra-9-285...,https://compujordan.com/image/cache/catalog/pr...,569.0,569.0,The Intel Core Ultra 9 285K (Series 2) is one ...,"gaming components, components, gaming, compute...",,intel intel core ultra 9 285k (series 2) – 24-...
4,RTX 5070 Ti 16G SHADOW 3X OC,msi geforce rtx 5070 ti 16g shadow 3x oc – 16g...,msi,Compu Jordan,https://compujordan.com/msi-geforce-rtx-5070-t...,https://compujordan.com/image/cache/catalog/pr...,789.0,789.0,The MSI GeForce RTX 5070 Ti 16G SHADOW 3X OC i...,"gaming components, components, gaming, compute...",,msi msi geforce rtx 5070 ti 16g shadow 3x oc –...


In [310]:
d = preprocess_file(file)

In [306]:
d.head()

,product_code,product_name,brand,store,url,image_url,price,original_price,product_description,categories,specs,model_text
0,D501MER,asus expertcenter d5 mini tower (d501mer) busi...,asus,Oriental Store,https://os-jo.com/asus-expertcenter-d5-mini-to...,https://os-jo.com/image/cache/catalog/products...,599.0,599.0,No Description,"apple, gaming pc, gaming laptop and gaming acc...","core i7, 14th generation, i7-14700, up to 5.4g...",asus asus expertcenter d5 mini tower (d501mer)...
1,B5TQ8AA,hp omnidesk desktop pc (m02-0234) amd ryzen 7 ...,hp,Oriental Store,https://os-jo.com/hp-omnidesk-desktop-pc-m02-0...,https://os-jo.com/image/cache/catalog/products...,629.0,629.0,No Description,"apple, gaming pc, gaming laptop and gaming acc...","amd, ryzen 7, 8700g, up to 5.1ghz, 16mb, 8, 16...",hp hp omnidesk desktop pc (m02-0234) amd ryzen...
2,BE8B1AA,hp omen 35l gaming pc (dt gt16-0344) (pre-buil...,hp,Oriental Store,https://os-jo.com/hp-omen-35l-gaming-pc-dt-gt1...,https://os-jo.com/image/cache/catalog/products...,1169.0,1169.0,No Description,"apple, gaming pc, gaming laptop and gaming acc...","amd ryzen 7 8700f, 1x16gb ddr5 supports up to...",hp hp omen 35l gaming pc (dt gt16-0344) (pre-b...
3,BC4K6AA,hp omen 16l gaming pc (dt tg03-0154) (pre-buil...,hp,Oriental Store,https://os-jo.com/hp-omen-16l-gaming-pc-dt-tg0...,https://os-jo.com/image/cache/catalog/products...,929.0,929.0,No Description,"apple, gaming pc, gaming laptop and gaming acc...","intel core i5 14400f, 1x16gb ddr5 supports up...",hp hp omen 16l gaming pc (dt tg03-0154) (pre-b...
4,1M-285BJO-B3100UXX,"msi cubi 1m mini pc, portable mini pc desktop,...",msi,Oriental Store,https://os-jo.com/msi-cubi-1m-mini-pc-portable...,https://os-jo.com/image/cache/catalog/products...,259.0,259.0,No Description,"apple, gaming pc, gaming laptop and gaming acc...","core 3, intel core (series 1), 100u, up to 4.7...","msi msi cubi 1m mini pc, portable mini pc desk..."


In [307]:
d['model_text'][0]

'asus asus expertcenter d5 mini tower (d501mer) business desktop 14th gen intel core i7-14700, 8gb ddr5 memory, intel uhd graphics, 512gb m.2 nvme gen 4 ssd, w/ 180w 80+ bronze power supply core i7, 14th generation, i7-14700, up to 5.4ghz, 33mb, 20 (8p+12e), 28 threads, 8gb (1 x 8gb), ddr5, m.2 pcie nvme ssd pcie 4.0, 512gb, intel, intel® uhd graphics 770, shared memory, integrated memory'

In [215]:
len(d['model_text'][0])

1078

In [126]:
df.head()

,product_code,product_name,brand,store,url,image_url,price,original_price,currency,in_stock,product_description,categories,specs
0,D501MER,ASUS ExpertCenter D5 Mini Tower (D501MER) Busi...,ASUS,Oriental Store,https://os-jo.com/asus-expertcenter-d5-mini-to...,https://os-jo.com/image/cache/catalog/products...,599.0,NaN,JOD,True,Unknown,"{'Apple', 'Lenovo', 'Computer Systems', 'Dell'...","{'Processor Type': 'Core i7', 'Processor Gener..."
1,B5TQ8AA,HP OmniDesk Desktop PC (M02-0234) AMD Ryzen 7 ...,HP,Oriental Store,https://os-jo.com/hp-omnidesk-desktop-pc-m02-0...,https://os-jo.com/image/cache/catalog/products...,629.0,NaN,JOD,True,Unknown,"{'Apple', 'Lenovo', 'Computer Systems', 'Dell'...","{'Processor Type': 'AMD', 'Processor Generatio..."
2,BE8B1AA,HP OMEN 35L Gaming PC (DT GT16-0344) (Pre-Buil...,HP,Oriental Store,https://os-jo.com/hp-omen-35l-gaming-pc-dt-gt1...,https://os-jo.com/image/cache/catalog/products...,1169.0,NaN,JOD,True,Unknown,"{'Apple', 'Lenovo', 'Computer Systems', 'Dell'...","{'Processor': 'AMD RYZEN 7 8700F', 'Motherboar..."
3,BC4K6AA,HP OMEN 16L Gaming PC (DT TG03-0154) (Pre-Buil...,HP,Oriental Store,https://os-jo.com/hp-omen-16l-gaming-pc-dt-tg0...,https://os-jo.com/image/cache/catalog/products...,929.0,NaN,JOD,True,Unknown,"{'Apple', 'Lenovo', 'Computer Systems', 'Dell'...","{'Processor': 'INTEL CORE I5 14400F', 'Motherb..."
4,1M-285BJO-B3100UXX,"MSI Cubi 1M Mini PC, Portable Mini PC Desktop,...",MSI,Oriental Store,https://os-jo.com/msi-cubi-1m-mini-pc-portable...,https://os-jo.com/image/cache/catalog/products...,259.0,NaN,JOD,True,Unknown,"{'Apple', 'Lenovo', 'Computer Systems', 'Dell'...","{'Processor Type': 'Core 3', 'Processor Genera..."


In [219]:
df['categories'][0]

"{'Apple', 'Lenovo', 'Computer Systems', 'Dell', 'Gaming PC', 'AMD', 'Gaming PC, Gaming Laptop and Gaming Accessories', 'Desktops Computer', 'Intel', 'MSI', 'HP', 'ASUS'}"

In [222]:
r = ast.literal_eval(df['specs'][0])

In [283]:
l = df['specs'][0]

In [295]:
l

"{'Processor Type': 'Core i7', 'Processor Generation': '14th Generation', 'Processor Family': 'i7-14700', 'Processor Speed': 'Up To 5.4GHz', 'Processor Cache': '33MB', 'Processor Cores': '20 (8P+12E)', 'Processor Threads': '28 Threads', 'Memory Size': '8GB (1 X 8GB)', 'Memory Type': 'DDR5', 'Number of Memory Slot': '2', 'Storage Technology': 'M.2 PCIe NVMe SSD PCIe 4.0', 'Storage Capacity': '512GB', 'Graphic Manufacturer': 'Intel', 'Graphic Chip-set Model': 'Intel® UHD Graphics 770', 'Graphic Memory Size': 'Shared Memory', 'Graphic Memory Type': 'Integrated Memory', 'Optical Drive': 'No', 'Ports': '*Front I/O Ports  1x Smart card reader  1x 2 in 1 card reader SD / MMC  2x USB 3.2 Gen 2 Type-A  2x USB 2.0 Type-A  1x 3.5mm combo audio jack  1x Headphone  *Rear I/O Ports  2x USB 3.2 Gen 1 Type-A  2x USB 2.0 Type-A  1x Padlock loop  1x Kensington lock  1x 7.1 channel audio (microphone  line-out  Line-in)  1x VGA Port  1x HDMI 1.4  1x RJ45 Gigabit Ethernet', 'Audio': 'High Definition 7.1 Ch

In [288]:
clean_specs(l)

'28 threads, 2, 8gb (1 x 8gb), 33mb, intel® uhd graphics 770, shared memory, intel, 20 (8p+12e), m.2 pcie nvme ssd pcie 4.0, core i7, 512gb, 14th generation, integrated memory, ddr5, i7-14700, up to 5.4ghz'

In [234]:
key_specs = {'processor',
                    'memory',
                    'storage',
                    'graphic'}

In [ ]:
set.isdisjoint()

In [257]:
specs = set()
for k, v in r.items():
    k_set = set(k.lower().split())
    if not k_set.isdisjoint(key_specs):
        specs.add(v)

In [258]:
specs

{'14th Generation',
 '2',
 '20 (8P+12E)',
 '28 Threads',
 '33MB',
 '512GB',
 '8GB (1 X 8GB)',
 'Core i7',
 'DDR5',
 'Integrated Memory',
 'Intel',
 'Intel® UHD Graphics 770',
 'M.2 PCIe NVMe SSD PCIe 4.0',
 'Shared Memory',
 'Up To 5.4GHz',
 'i7-14700'}

In [238]:
q = list(r.items())[0][0]

In [242]:
set(q.lower().split()) & key_specs != None

True

In [127]:
df.describe(include='all')

,product_code,product_name,brand,store,url,image_url,price,original_price,currency,in_stock,product_description,categories,specs
count,5041,5041,5041,5041,5041,5034,5038.000000,1117.000000,5041,5041,4067,5041,5041
unique,4281,5034,75,1,5041,4419,NaN,NaN,2,2,459,81,3846
top,OS BUILD,Unknown,ASUS,Oriental Store,https://os-jo.com/asus-expertcenter-d5-mini-to...,https://os-jo.com/image/cache/catalog/products...,NaN,NaN,JOD,False,Unknown,"{'Gaming PC', 'AMD', 'Gaming PC, Gaming Laptop...",{}
freq,184,3,428,5041,1,23,NaN,NaN,5038,3796,3501,377,734
mean,NaN,NaN,NaN,NaN,NaN,NaN,359.976372,519.400179,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,772.243669,648.346907,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,10.000000,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,49.000000,95.000000,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,129.000000,245.000000,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,379.000000,729.000000,NaN,NaN,NaN,NaN,NaN


In [73]:
df['price'] = df['price'].astype(object)

In [74]:
df['original_price'] = df['original_price'].astype(object)

In [128]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5041 entries, 0 to 5040
Data columns (total 13 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   product_code         5041 non-null   object 
 1   product_name         5041 non-null   object 
 2   brand                5041 non-null   object 
 3   store                5041 non-null   object 
 4   url                  5041 non-null   object 
 5   image_url            5034 non-null   object 
 6   price                5038 non-null   float64
 7   original_price       1117 non-null   float64
 8   currency             5041 non-null   object 
 9   in_stock             5041 non-null   bool   
 10  product_description  4067 non-null   object 
 11  categories           5041 non-null   object 
 12  specs                5041 non-null   object 
dtypes: bool(1), float64(2), object(10)
memory usage: 516.9+ KB


In [76]:
df.describe()

,product_code,product_name,brand,store,url,image_url,price,original_price,currency,in_stock,product_description,categories,specs
count,5041,5041,5041,5041,5041,5034,5038.0,1117.0,5041,5041,4067,5041,5041
unique,4281,5034,75,1,5041,4419,358.0,243.0,2,2,459,81,3846
top,OS BUILD,Unknown,ASUS,Oriental Store,https://os-jo.com/asus-expertcenter-d5-mini-to...,https://os-jo.com/image/cache/catalog/products...,99.0,99.0,JOD,False,Unknown,"{'Gaming PC', 'AMD', 'Gaming PC, Gaming Laptop...",{}
freq,184,3,428,5041,1,23,133.0,31.0,5038,3796,3501,377,734


In [82]:
valid_df = df[df['in_stock'] == True].copy()

In [83]:
valid_df.describe()

,product_code,product_name,brand,store,url,image_url,price,original_price,currency,in_stock,product_description,categories,specs
count,1245,1245,1245,1245,1245,1245,1245.0,111.0,1245,1245,836,1245,1245
unique,1143,1245,53,1,1245,1119,185.0,72.0,1,1,206,46,797
top,OS BUILD,ASUS ExpertCenter D5 Mini Tower (D501MER) Busi...,Unknown,Oriental Store,https://os-jo.com/asus-expertcenter-d5-mini-to...,https://os-jo.com/image/cache/catalog/products...,25.0,199.0,JOD,True,Unknown,"{'Cooler Master', 'Keyboard & Mouse', 'GLORIOU...",{}
freq,21,1,146,1245,1,13,42.0,5.0,1245,1245,603,99,353


In [124]:
# pd.set_option('future.no_silent_downcasting', True)

In [85]:
valid_df['original_price'] = valid_df['original_price'].fillna(valid_df['price'])

In [86]:
valid_df['original_price'][0]

599.0

In [88]:
valid_df.describe()

,product_code,product_name,brand,store,url,image_url,price,original_price,currency,in_stock,product_description,categories,specs
count,1245,1245,1245,1245,1245,1245,1245.0,1245.0,1245,1245,836,1245,1245
unique,1143,1245,53,1,1245,1119,185.0,181.0,1,1,206,46,797
top,OS BUILD,ASUS ExpertCenter D5 Mini Tower (D501MER) Busi...,Unknown,Oriental Store,https://os-jo.com/asus-expertcenter-d5-mini-to...,https://os-jo.com/image/cache/catalog/products...,25.0,69.0,JOD,True,Unknown,"{'Cooler Master', 'Keyboard & Mouse', 'GLORIOU...",{}
freq,21,1,146,1245,1,13,42.0,41.0,1245,1245,603,99,353


In [ ]:
str.islow

In [96]:
valid_df[~valid_df['product_code'].str.isupper()]

,product_code,product_name,brand,store,url,image_url,price,original_price,currency,in_stock,product_description,categories,specs
5,Dell Tower ECT1250,"Dell Tower ECT1250, Micro Tower Business Deskt...",Dell,Oriental Store,https://os-jo.com/dell-tower-ect1250-micro-tow...,https://os-jo.com/image/cache/catalog/products...,349.0,349.0,JOD,True,Unknown,"{'Apple', 'Lenovo', 'Computer Systems', 'Dell'...","{'Processor Type': 'Core i3', 'Processor Gener..."
7,OptiPlex Tower Plus 7020,Dell OptiPlex Tower Plus 7020 Business Desktop...,Dell,Oriental Store,https://os-jo.com/dell-optiplex-tower-plus-702...,https://os-jo.com/image/cache/catalog/products...,529.0,529.0,JOD,True,Unknown,"{'Apple', 'Lenovo', 'Computer Systems', 'Dell'...","{'Processor Type': 'Core i5', 'Processor Gener..."
8,Mac Pro Tower,"Apple Mac Pro Tower Enclosure, Apple M2 Ultra ...",Apple,Oriental Store,https://os-jo.com/apple-mac-pro-tower-enclosur...,https://os-jo.com/image/cache/catalog/products...,5999.0,5999.0,JOD,True,Unknown,"{'Apple', 'Lenovo', 'Computer Systems', 'Dell'...","{'Processor Type': 'Apple M2 Ultra', 'Processo..."
10,OptiPlex Tower Plus 7020,Dell OptiPlex Tower Plus 7020 Business Desktop...,Dell,Oriental Store,https://os-jo.com/dell-optiplex-tower-plus-702...,https://os-jo.com/image/cache/catalog/products...,649.0,649.0,JOD,True,Unknown,"{'Apple', 'Lenovo', 'Computer Systems', 'Dell'...","{'Processor Type': 'Core i7', 'Processor Gener..."
13,"Apple iMac 24"" M3 Chip","Apple iMac 24"" with Retina 4.5K display Apple ...",Apple,Oriental Store,https://os-jo.com/apple-imac-24-with-retina-4-...,https://os-jo.com/image/cache/catalog/products...,1589.0,1589.0,JOD,True,Unknown,"{'Apple', 'Lenovo', 'Computer Systems', 'Dell'...","{'Processor Type': 'Apple M3', 'Processor Gene..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
4949,Logitech G304 (White),Logitech G304 Lightspeed Wireless (2.4GHz) Gam...,Logitech,Oriental Store,https://os-jo.com/logitech-g304-lightspeed-wir...,https://os-jo.com/image/cache/catalog/products...,30.0,30.0,JOD,True,Unknown,"{'Logitech', 'RAZER', 'Gaming PC, Gaming Lapto...",{'Full Product Specifications': '• PHYSICAL SP...
4950,Logitech G304 (Black),Logitech G304 Lightspeed Wireless (2.4GHz) Gam...,Logitech,Oriental Store,https://os-jo.com/logitech-g304-lightspeed-wir...,https://os-jo.com/image/cache/catalog/products...,30.0,30.0,JOD,True,Unknown,"{'Logitech', 'RAZER', 'Gaming PC, Gaming Lapto...",{'Full Product Specifications': '• PHYSICAL SP...
4951,Logitech G102 White,"Logitech G102 Customizable Wired RGB Mouse, 6 ...",Logitech,Oriental Store,https://os-jo.com/logitech-g102-customizable-r...,https://os-jo.com/image/cache/catalog/products...,17.0,17.0,JOD,True,Unknown,"{'Logitech', 'RAZER', 'Gaming PC, Gaming Lapto...",{'Full Product Specifications': '• PHYSICAL SP...
4953,Logitech BRIO 4K PRO,Logitech BRIO 4K Stream Edition HDR & Autofocu...,Logitech,Oriental Store,https://os-jo.com/logitech-brio-4k-stream-edit...,https://os-jo.com/image/cache/catalog/products...,159.0,159.0,JOD,True,Unknown,"{'Logitech', 'Gaming PC, Gaming Laptop and Gam...",{'Full Product Specifications': '•DIMENSIONS:H...


In [98]:
valid_df['product_name'] = valid_df['product_name'].str.lower()

In [100]:
valid_df['brand'] = valid_df['brand'].str.lower()

In [101]:
valid_df.head()

,product_code,product_name,brand,store,url,image_url,price,original_price,currency,in_stock,product_description,categories,specs
0,D501MER,asus expertcenter d5 mini tower (d501mer) busi...,asus,Oriental Store,https://os-jo.com/asus-expertcenter-d5-mini-to...,https://os-jo.com/image/cache/catalog/products...,599.0,599.0,JOD,True,Unknown,"{'Apple', 'Lenovo', 'Computer Systems', 'Dell'...","{'Processor Type': 'Core i7', 'Processor Gener..."
1,B5TQ8AA,hp omnidesk desktop pc (m02-0234) amd ryzen 7 ...,hp,Oriental Store,https://os-jo.com/hp-omnidesk-desktop-pc-m02-0...,https://os-jo.com/image/cache/catalog/products...,629.0,629.0,JOD,True,Unknown,"{'Apple', 'Lenovo', 'Computer Systems', 'Dell'...","{'Processor Type': 'AMD', 'Processor Generatio..."
2,BE8B1AA,hp omen 35l gaming pc (dt gt16-0344) (pre-buil...,hp,Oriental Store,https://os-jo.com/hp-omen-35l-gaming-pc-dt-gt1...,https://os-jo.com/image/cache/catalog/products...,1169.0,1169.0,JOD,True,Unknown,"{'Apple', 'Lenovo', 'Computer Systems', 'Dell'...","{'Processor': 'AMD RYZEN 7 8700F', 'Motherboar..."
3,BC4K6AA,hp omen 16l gaming pc (dt tg03-0154) (pre-buil...,hp,Oriental Store,https://os-jo.com/hp-omen-16l-gaming-pc-dt-tg0...,https://os-jo.com/image/cache/catalog/products...,929.0,929.0,JOD,True,Unknown,"{'Apple', 'Lenovo', 'Computer Systems', 'Dell'...","{'Processor': 'INTEL CORE I5 14400F', 'Motherb..."
4,1M-285BJO-B3100UXX,"msi cubi 1m mini pc, portable mini pc desktop,...",msi,Oriental Store,https://os-jo.com/msi-cubi-1m-mini-pc-portable...,https://os-jo.com/image/cache/catalog/products...,259.0,259.0,JOD,True,Unknown,"{'Apple', 'Lenovo', 'Computer Systems', 'Dell'...","{'Processor Type': 'Core 3', 'Processor Genera..."


In [103]:
t = valid_df['specs'].loc[0]

In [110]:
t

"{'Processor Type': 'Core i7', 'Processor Generation': '14th Generation', 'Processor Family': 'i7-14700', 'Processor Speed': 'Up To 5.4GHz', 'Processor Cache': '33MB', 'Processor Cores': '20 (8P+12E)', 'Processor Threads': '28 Threads', 'Memory Size': '8GB (1 X 8GB)', 'Memory Type': 'DDR5', 'Number of Memory Slot': '2', 'Storage Technology': 'M.2 PCIe NVMe SSD PCIe 4.0', 'Storage Capacity': '512GB', 'Graphic Manufacturer': 'Intel', 'Graphic Chip-set Model': 'Intel® UHD Graphics 770', 'Graphic Memory Size': 'Shared Memory', 'Graphic Memory Type': 'Integrated Memory', 'Optical Drive': 'No', 'Ports': '*Front I/O Ports  1x Smart card reader  1x 2 in 1 card reader SD / MMC  2x USB 3.2 Gen 2 Type-A  2x USB 2.0 Type-A  1x 3.5mm combo audio jack  1x Headphone  *Rear I/O Ports  2x USB 3.2 Gen 1 Type-A  2x USB 2.0 Type-A  1x Padlock loop  1x Kensington lock  1x 7.1 channel audio (microphone  line-out  Line-in)  1x VGA Port  1x HDMI 1.4  1x RJ45 Gigabit Ethernet', 'Audio': 'High Definition 7.1 Ch

In [108]:
len(t)

1271

In [106]:
type(t)a


str

In [113]:
e = ast.literal_eval(t)

In [121]:
q = ' '.join(list(e.values()))

In [122]:
len(q)

745

In [123]:
q

'Core i7 14th Generation i7-14700 Up To 5.4GHz 33MB 20 (8P+12E) 28 Threads 8GB (1 X 8GB) DDR5 2 M.2 PCIe NVMe SSD PCIe 4.0 512GB Intel Intel® UHD Graphics 770 Shared Memory Integrated Memory No *Front I/O Ports  1x Smart card reader  1x 2 in 1 card reader SD / MMC  2x USB 3.2 Gen 2 Type-A  2x USB 2.0 Type-A  1x 3.5mm combo audio jack  1x Headphone  *Rear I/O Ports  2x USB 3.2 Gen 1 Type-A  2x USB 2.0 Type-A  1x Padlock loop  1x Kensington lock  1x 7.1 channel audio (microphone  line-out  Line-in)  1x VGA Port  1x HDMI 1.4  1x RJ45 Gigabit Ethernet High Definition 7.1 Channel Audio Free DOS RJ45 Gigabit Ethernet 10/100/1000  Wi-Fi 6(802.11ax) (Dual band) 2*2 + Bluetooth® 5.4 Wireless Card Wired Keyboard + Mouse 1 Year Intel® B760 Chipset'